# 04 — Modelado Predictivo y Evaluación — Titanic

Este cuaderno implementa las Partes 10 a 17 del taller: formulación, baselines, prevención de fuga de datos con `Pipeline`, validación cruzada, diagnóstico de errores, ajuste de hiperparámetros, interpretabilidad, ablación, evaluación única en `X_test` y prueba de humo sobre datos sin etiqueta.

## Parte 10 — Formulación, División y Baselines (Pasos 39–42)

In [ ]:
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.calibration import calibration_curve
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    brier_score_loss,
    classification_report,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_predict,
    cross_val_score,
    learning_curve,
)

from flows.common.paths import BRONZE, BRONZE_UNLABELED_NAME, MODELS

ml = importlib.import_module("flows.05_ml_models")
de = importlib.import_module("flows.01_data_engineering")

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 5)

X, y = ml.load_dataset()
print("Dataset cargado:", X.shape, "| Tasa supervivencia:", round(y.mean(), 4))

In [ ]:
# Paso 41: División estratificada entrenamiento / prueba (X_test se guarda hasta la Parte 16)
X_train, X_test, y_train, y_test = ml.split(X, y)
print(f"Train: {X_train.shape}, tasa={y_train.mean():.3f} | Test: {X_test.shape}, tasa={y_test.mean():.3f}")

# Paso 42: Baselines sobre entrenamiento
cv = StratifiedKFold(n_splits=ml.settings.cv_folds, shuffle=True, random_state=ml.settings.random_state)

dummy_scores = cross_val_score(DummyClassifier(strategy="most_frequent"), X_train, y_train, cv=cv, scoring="accuracy")
gender_pred_train = (X_train["Sex"] == "female").astype(int)

print(f"Baseline Dummy (most_frequent) CV accuracy: {dummy_scores.mean():.3f} ± {dummy_scores.std():.3f}")
print(
    f"Baseline Regla de Género (train) -> "
    f"Accuracy: {accuracy_score(y_train, gender_pred_train):.3f}, "
    f"Precision: {precision_score(y_train, gender_pred_train):.3f}, "
    f"Recall: {recall_score(y_train, gender_pred_train):.3f}, "
    f"F1: {f1_score(y_train, gender_pred_train):.3f}"
)

## Parte 11 — Preprocesamiento sin Fuga de Datos (Pasos 43–46)

In [ ]:
# Paso 43: Demostración de fuga de datos al imputar con todo el dataset vs solo train
mean_all_f = X.loc[X["Sex"] == "female", "Age"].mean()
mean_tr_f = X_train.loc[X_train["Sex"] == "female", "Age"].mean()
print(f"Edad media mujeres (TODO X): {mean_all_f:.3f} vs (SOLO X_train): {mean_tr_f:.3f} | Dif: {abs(mean_all_f - mean_tr_f):.3f}")

# Paso 45-46: Comparación de imputación por (Sex) vs (Sex, Pclass) y verificación del preprocesador
pipe_check = ml.build_pipeline(DummyClassifier())
pre = pipe_check[:-1].fit(X_train)
Z_val = pre.transform(X_train.iloc[:10])
print("¿Contiene NaN después del preprocesador?:", np.isnan(Z_val).any())
print("Columnas transformadas:", pre["prep"].get_feature_names_out())

## Parte 12 — Modelos Candidatos y Validación Cruzada (Pasos 47–50)

In [ ]:
cv_summary = ml.cross_validate_candidates(X_train, y_train)
display(cv_summary[["model", "test_roc_auc_mean", "test_roc_auc_std", "train_roc_auc_mean", "overfit_gap_roc_auc", "test_accuracy_mean", "test_f1_mean"]].round(4))

# Boxplot de ROC-AUC por pliegue para cada candidato
fold_records = []
for name, model in ml.get_candidates(ml.settings.random_state).items():
    pipe = ml.build_pipeline(model)
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    for sc in scores:
        fold_records.append({"model": name, "roc_auc": sc})

plt.figure(figsize=(9, 4))
sns.boxplot(data=pd.DataFrame(fold_records), x="model", y="roc_auc")
plt.title("Distribución de ROC-AUC en Validación Cruzada (5 pliegues)")
plt.show()

## Parte 13 — Evaluación en Profundidad Fuera de Muestra (Pasos 51–55)

In [ ]:
finalists = ["logreg", "forest", "hgb"]
oof_probas = {}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name in finalists:
    pipe = ml.build_pipeline(ml.get_candidates(ml.settings.random_state)[name])
    proba = cross_val_predict(pipe, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
    oof_probas[name] = proba
    RocCurveDisplay.from_predictions(y_train, proba, name=name, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_train, proba, name=name, ax=axes[1])

axes[0].set_title("Curvas ROC (Fuera de Muestra)")
axes[1].set_title("Curvas Precisión-Recall (Fuera de Muestra)")
plt.tight_layout()
plt.show()

# Matriz de confusión y reporte para el finalista principal (hgb)
proba_hgb = oof_probas["hgb"]
pred_hgb = (proba_hgb >= 0.5).astype(int)
ConfusionMatrixDisplay.from_predictions(y_train, pred_hgb)
plt.title("Matriz de Confusión OOF — HistGradientBoosting (umbral=0.5)")
plt.show()
print(classification_report(y_train, pred_hgb))

In [ ]:
# Paso 53: Calibración y búsqueda de umbral óptimo en OOF
prob_true, prob_pred = calibration_curve(y_train, proba_hgb, n_bins=8)
plt.figure(figsize=(6, 5))
plt.plot(prob_pred, prob_true, marker="o", label="HGB")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Calibración perfecta")
plt.xlabel("Probabilidad predicha")
plt.ylabel("Fracción real de positivos")
plt.title(f"Curva de Calibración (Brier={brier_score_loss(y_train, proba_hgb):.3f})")
plt.legend()
plt.show()

prec, rec, thresh = precision_recall_curve(y_train, proba_hgb)
f1s = 2 * (prec[:-1] * rec[:-1]) / (prec[:-1] + rec[:-1] + 1e-12)
best_idx = int(np.argmax(f1s))
print(f"Umbral máximo F1 en OOF: {thresh[best_idx]:.3f} (F1={f1s[best_idx]:.3f}) vs umbral=0.500 (F1={f1_score(y_train, pred_hgb):.3f})")

In [ ]:
# Paso 54 y 55: Análisis de errores por subgrupo y curva de aprendizaje
err_df = X_train.copy()
err_df["y_true"] = y_train
err_df["proba"] = proba_hgb
err_df["pred"] = pred_hgb
err_df["error"] = (err_df["pred"] != err_df["y_true"]).astype(int)

print("Exactitud por Sexo:")
display(err_df.groupby("Sex").apply(lambda g: pd.Series({"n": len(g), "accuracy": (g["pred"] == g["y_true"]).mean()})))
print("Exactitud por Clase:")
display(err_df.groupby("Pclass").apply(lambda g: pd.Series({"n": len(g), "accuracy": (g["pred"] == g["y_true"]).mean()})))

err_df["confianza_error"] = np.abs(err_df["proba"] - err_df["y_true"])
print("Top 5 errores de mayor confianza:")
display(err_df[err_df["error"] == 1].nlargest(5, "confianza_error")[["Pclass", "Sex", "Age", "Fare", "TTitulo", "y_true", "proba"]])

pipe_hgb = ml.build_pipeline(ml.get_candidates(ml.settings.random_state)["hgb"])
sizes, tr_sc, va_sc = learning_curve(pipe_hgb, X_train, y_train, cv=cv, train_sizes=np.linspace(0.2, 1.0, 5), scoring="roc_auc", n_jobs=-1)
plt.figure(figsize=(8, 4))
plt.plot(sizes, tr_sc.mean(axis=1), "o-", label="Entrenamiento")
plt.plot(sizes, va_sc.mean(axis=1), "o-", label="Validación CV")
plt.fill_between(sizes, va_sc.mean(axis=1) - va_sc.std(axis=1), va_sc.mean(axis=1) + va_sc.std(axis=1), alpha=0.15)
plt.title("Curva de Aprendizaje — HistGradientBoosting")
plt.xlabel("Tamaño del conjunto de entrenamiento")
plt.ylabel("ROC-AUC")
plt.legend()
plt.show()

## Parte 14 — Ajuste de Hiperparámetros (Pasos 56–58)

In [ ]:
grids = ml.get_param_grids()
best_estimators = {}
for name in finalists:
    base_model = ml.get_candidates(ml.settings.random_state)[name]
    gs = GridSearchCV(ml.build_pipeline(base_model), grids[name], scoring=ml.PRIMARY_METRIC, cv=cv, n_jobs=-1, refit=True)
    gs.fit(X_train, y_train)
    best_estimators[name] = gs.best_estimator_
    print(f"{name} -> best ROC-AUC CV: {gs.best_score_:.4f} | params: {gs.best_params_}")

# Paso 58: Comparación pareada pliegue a pliegue (HGB por defecto vs ajustado)
default_folds = cross_val_score(pipe_hgb, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
tuned_folds = cross_val_score(best_estimators["hgb"], X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
diff = tuned_folds - default_folds
print(f"Diferencia pareada (ajustado - defecto): {diff.mean():+.4f} ± {diff.std():.4f}")

## Parte 15 — Interpretación y Ablación de Variables (Pasos 59–61)

In [ ]:
# Paso 59: Coeficientes y Odds Ratios de Regresión Logística
pipe_lr = best_estimators["logreg"]
feat_names = pipe_lr["prep"].get_feature_names_out()
coefs = pipe_lr["model"].coef_[0]
coef_df = pd.DataFrame({
    "feature": feat_names,
    "coef": coefs,
    "abs_coef": np.abs(coefs),
    "odds_ratio": np.exp(coefs),
}).sort_values("abs_coef", ascending=False)
display(coef_df.head(10).round(4))

# Paso 60: Importancia por permutación sobre partición de validación interna
X_tr_sub, X_val_sub, y_tr_sub, y_val_sub = ml.split(X_train, y_train, test_size=0.25)
pipe_perm = ml.train_final(X_tr_sub, y_tr_sub)
used_cols = ml.NUM_FEATURES + ml.CAT_FEATURES
perm = permutation_importance(pipe_perm, X_val_sub[used_cols], y_val_sub, scoring="roc_auc", n_repeats=30, random_state=42, n_jobs=-1)
perm_df = pd.DataFrame({"feature": used_cols, "importance_mean": perm.importances_mean, "importance_std": perm.importances_std}).sort_values("importance_mean", ascending=False)
display(perm_df.round(4))

In [ ]:
# Paso 61: Ablación de grupos de variables
ablations = {
    "A) Base (NUM + CAT)": (ml.NUM_FEATURES, ml.CAT_FEATURES),
    "B) Base + Cabina (HasCabin, TCabin)": (ml.NUM_FEATURES + ml.CABIN_NUM, ml.CAT_FEATURES + ml.CABIN_CAT),
    "C) Base sin TTitulo": (ml.NUM_FEATURES, [c for c in ml.CAT_FEATURES if c != "TTitulo"]),
    "D) Base sin Sex": (ml.NUM_FEATURES, [c for c in ml.CAT_FEATURES if c != "Sex"]),
}
abl_rows = []
base_auc = None
for label, (n_cols, c_cols) in ablations.items():
    modelo = ml.get_candidates(ml.settings.random_state)[ml.FINAL_ESTIMATOR]
    p = ml.build_pipeline(modelo, num_features=n_cols, cat_features=c_cols).set_params(**ml.FINAL_PARAMS)
    sc = cross_val_score(p, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    if base_auc is None:
        base_auc = sc.mean()
    abl_rows.append({"configuracion": label, "roc_auc_mean": sc.mean(), "roc_auc_std": sc.std(), "delta_vs_A": sc.mean() - base_auc})

display(pd.DataFrame(abl_rows).round(4))

## Partes 16 y 17 — Evaluación Única en Test y Prueba de Humo (Pasos 62–69)

In [ ]:
# Paso 62: Evaluación ÚNICA sobre X_test
pipe_eval = ml.train_final(X_train, y_train)
test_metrics = ml.evaluate_on_test(pipe_eval, X_test, y_test, threshold=ml.THRESHOLD)
print("Métricas finales en conjunto de prueba (n=179):")
display(pd.Series(test_metrics))

# Paso 69: Prueba de humo sobre pasajeros sin etiqueta (Bronze unlabeled)
pipe_saved, meta = ml.load_model(MODELS / f"{ml.MODEL_NAME}.joblib")
nuevos_raw = pd.read_csv(BRONZE / BRONZE_UNLABELED_NAME)
nuevos_silver = (
    nuevos_raw.pipe(de.cast_types)
    .pipe(de.add_person)
    .pipe(de.add_cabin_features)
    .pipe(de.add_family_features)
    .pipe(de.add_title)
    .pipe(de.consolidate_titles)
    .pipe(de.impute_embarked)
    .pipe(de.impute_age)
)
X_nuevos = nuevos_silver[ml.CANDIDATE_COLUMNS].copy()
X_nuevos["Age"] = nuevos_silver["AgeOriginal"]

preds_nuevos = ml.predict(pipe_saved, X_nuevos, threshold=meta["threshold"])
print("Predicciones sobre 418 pasajeros sin etiqueta -> nulos:", preds_nuevos.isna().sum().to_dict())
display(preds_nuevos.describe().round(4))
print("Distribución de clases predichas (0/1):")
display(preds_nuevos["pred"].value_counts(normalize=True).round(4))